# ch02 — Gen1 통계: PCA-T²/SPE

SMD(로컬 data/ 필요) 또는 합성 다변량 데이터에 적용.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
# SMD가 있으면 실데이터, 없으면 합성 다변량
try:
    from tsad_forge.data.registry import load_dataset
    ds = load_dataset("smd", machine="machine-1-1")
    print("using SMD machine-1-1")
except FileNotFoundError:
    ds = generate_synthetic(n_dims=8, n_events=6, seed=2)
    print("using synthetic (run `tsad-forge download smd` for real data)")

In [ ]:
from tsad_forge.evaluation.protocol import zscore_normalize
from tsad_forge.models.registry import get_model

train, test = zscore_normalize(ds.train, ds.test)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, mode in zip(axes, ["t2", "spe", "combined"]):
    scores = get_model("pca_t2spe", mode=mode).fit(train).score(test)
    ax.plot(scores, lw=0.6)
    ax.fill_between(np.arange(len(scores)), *ax.get_ylim(),
                    where=ds.labels.astype(bool), alpha=0.25, color="red")
    ax.set_title(f"PCA-{mode.upper()} — T²(주성분 공간) vs SPE(잔차 공간)는 다른 이상을 잡는다")
plt.tight_layout()